# QA to compare with MEM1

In [1]:
import pandas as pd
from pprint import pprint
import pdb
import numpy as np
import plotly.graph_objects as go
import json
import pickle

def get_df_from_train_gen(file: str):
    raw_jl = open(file, 'r', encoding='utf-8').read()

    json_list = []
    one_method_to_split = raw_jl.strip().split("\n}")
    end_str = "\n}"
    if len(one_method_to_split) == 1: # detect other mode of data generation.
        one_method_to_split = raw_jl.strip().split("\n")
        end_str = "\n"

    for i, raw_json in enumerate(one_method_to_split):
        if raw_json.strip():
            json_list.append(json.loads(raw_json + end_str))
    df = pd.DataFrame(json_list)
    return df

def deduplicate_df(dfraw):
    odf = pd.concat([pd.DataFrame(dfraw.iloc[i].to_dict()) for i in range(len(dfraw))]) # concatenate all the evaluated batches. some questions may be duplicated across batches
    odf['questions']=odf.apply(lambda x: x.full_trajectory_strings.split('Answer the following questions: ')[1].split('\n')[0], axis=1) # extract questions 
    keep_uid_per_q = ( # one traj_uid per question (first seen)
        odf.groupby('questions')['traj_uid']
        .transform('first'))            # the chosen uid for that question
    odf_deduplicated = odf[odf['traj_uid'] == keep_uid_per_q]
    assert len(odf_deduplicated.groupby('traj_uid').size()) == len(odf_deduplicated.groupby('questions').size())
    return odf_deduplicated

eval_folder_path = "/nas/ucb/jbjorner3/dev/optimal-explorer-dev/verl-agent/checkpoints/verl_agent_alfworld/"

REPORTED_OBJS = [2,8,16]
ALL_NUM_OBJS = [1, 2, 4, 8, 16]
OBJS_TO_X = {1: 1, 2: 2, 4: 3.5, 8: 5, 16: 6.5}


In [26]:
#  MEM1
mem1_objectives_dfs ={}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_TrueGRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    mem1_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [2]:
mem1_seeds = {}
for seed in range(1,4):
    mem1_objectives_dfs ={}
    print("loading seed", seed)
    for num_objs in ALL_NUM_OBJS:
        dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed{str(seed)}_sc_False_belief_prompting_True_is_mem1_True_belief_len_pen_0GRPO_INSTRUCT_MEM1_SEED_{str(seed)}/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
        mem1_objectives_dfs[num_objs] = deduplicate_df(dfraw)
    mem1_seeds[seed] = mem1_objectives_dfs

loading seed 1
loading seed 2
loading seed 3


In [27]:
#ABBEL
temperature = '0.01'
length_penalty = '0.0'
abbel_objectives_dfs01 ={}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs01[num_objs] = deduplicate_df(dfraw)

In [41]:
# Zero-Shot ABBEL nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_TrueABBEL_ckpt__objectives_1_inference3
zabbel_objectives_dfs = {}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_TrueABBEL_ckpt__objectives_{str(num_objs)}_inference3/train_gen.txt")
    zabbel_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [25]:
#ABBEL Length Penalty
temperature = '0.01'
length_penalty = '0.01'
abbel_objectives_dfs_penalty01 = {}
for num_objs in [1,2,4,8,16]:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs_penalty01[num_objs] = deduplicate_df(dfraw)

In [103]:
#ABBEL Length Penalty 
abbel_objectives_dfs_penalty01s2 = {}
for num_objs in [1,2,4,8,16]:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_False_belief_prompting_TrueABBEL_LEN_PEN_0_01_ckpt_qwen2.5-7b-instruct_16sfr_seed2_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_0.01GRPO_INSTRUCT_SEED_2/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    abbel_objectives_dfs_penalty01s2[num_objs] = deduplicate_df(dfraw)

In [4]:
#Vanilla
vanilla_objectives_dfs = {}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path +f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_True_belief_prompting_False_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_True_belief_prompting_False_is_mem1_False_belief_len_pen_0GRPO_INSTRUCT/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    vanilla_objectives_dfs[num_objs] = deduplicate_df(dfraw)


In [157]:
#Vanilla fixed
vanilla2_objectives_dfs = {}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path +f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_True_belief_prompting_False_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_True_belief_prompting_False_is_mem1_False_belief_len_pen_0GRPO_INSTRUCT_VANILLA_nqbranch/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
    vanilla2_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [14]:
# Zero-Shot Vanilla
zvanilla_objectives_dfs = {}
for num_objs in ALL_NUM_OBJS:
    dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_0.01_256sfr_seed1_sc_True_belief_prompting_FalseVANILLA_ckpt__objectives_{str(num_objs)}_inference3/train_gen.txt")
    zvanilla_objectives_dfs[num_objs] = deduplicate_df(dfraw)

In [ ]:
# Old penalty runs
# temperature = '1.0'
# length_penalty = '0.1'
# abbel_objectives_dfs_penalty ={}
# for num_objs in [1, 2, 4, 8, 16]:
#     dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_4/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
#     abbel_objectives_dfs_penalty[num_objs] = deduplicate_df(dfraw)

# temperature = '1.0'
# length_penalty = '0.05'
# abbel_objectives_dfs_penalty05 = {}
# for num_objs in [1,2,4,8,16]:
#     dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_4/global_step_260_objectives_{str(num_objs)}_inference3/train_gen.txt")
#     abbel_objectives_dfs_penalty05[num_objs] = deduplicate_df(dfraw)

# Qualitative Look At Trajectories

In [ ]:
count = 0
for i, traj in vanilla_objectives_dfs[2].groupby('traj_uid'):
    count += 1
    if count == 1:#traj.rewards.max() > 3:
        break
print(traj.full_trajectory_strings.iloc[0].split('assistant\n', 1)[1])
traj.episode_rewards.max()

<think> To start, I need to clarify which film you're asking about for the first question. Since the film is not specified, I'll begin with the second question as it has a specific title: "Sabotage". </think>
<search> who was the screenwriter for Sabotage? </search>
user
<hint>You have 5 turns left.</hint>

Doc 1(Title: "Sabotage (2014 film)") Sabotage (2014 film) Sabotage is a 2014 American action thriller film co-written and directed by David Ayer and starring Arnold Schwarzenegger, Sam Worthington, Olivia Williams, Mireille Enos and Terrence Howard. The film was released in the United States on March 28, 2014. John ""Breacher"" Wharton is the leader of the Drug Enforcement Administration's Special Operations Team, which consists of James ""Monster"" Murray, Monster's wife Lizzy Murray, Joe ""Grinder"" Philips, Julius ""Sugar"" Edmonds, Eddie ""Neck"" Jordan, Tom ""Pyro"" Roberts, Bryce ""Tripod"" McNeely and ""Smoke"" Jennings. During a raid on a cartel warehouse, Smoke was killed, 

np.float64(0.0)

In [189]:
count = 0
for i, traj in zabbel_objectives_dfs[16].groupby('traj_uid'):
    count += 1
    if count ==1:#traj.episode_rewards.max() > 4:
        break
print(traj.full_trajectory_strings.iloc[0])#.split('assistant\n', 1)[1])
traj.episode_rewards.max()

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: Who was the director of Robosapien: Rebooted?; Who was the director of The Gift?; Who was the director of The Girl in Mourning?; Who was the director of Last Days?; Who was the director of La Notte?; Who was the director of Swamp Woman?; Who was the director of Cannibal Tours?; Who was the director of Mother?;

np.float64(5.0)

In [176]:
len("<answer> sternum; Philip Pullman; King George II; Jacinda Ardern; Sheffield United; Simply Red; First World War; Jacqui Oatley; Procol Harum; 1932-1933; Treasure Island; Václav Havel; Salvador Dalí; Laos </answer>".split(';'))


14

In [ ]:
count = 0
for i, traj in abbel_objectives_dfs_penalty01[16].groupby('traj_uid'):
    count += 1
    if count == 4:#traj.rewards.max() > 3:
        break
print(traj.sort_values(by='step').full_trajectory_strings.iloc[5])

In [108]:
count = 0
for i, traj in abbel_objectives_dfs_penalty01[16].groupby('traj_uid'):
    count += 1
    if count == 4:#traj.rewards.max() > 3:
        break
print(traj.sort_values(by='step').full_trajectory_strings.iloc[5])

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: What literary style describes the 1968 book that features a 1939 international Harvester school bus?; Is the building located at 200 West Street taller than the one at 888 7th Avenue?; who was born first Pierre Womé or Christian Poulsen ?; Where is the company owning Reliance Cricket Stadium ranked on the Fort

In [103]:
count = 0
for i, traj in abbel_objectives_dfs01[16].groupby('traj_uid'):
    count += 1
    if count == 12:#traj.rewards.max() > 3:
        break
print(traj.sort_values(by='step').full_trajectory_strings.iloc[5])

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: The principal author of the US Constitution and writer of over 1/3 of the Federalist Papers, which US President, the 4th, was CIC during the War of 1812?; Of what modern country is Sarajevo the captial of?; Pre restraining order(s), who did People magazine name as their first "Sexiest Man Alive", in 1985?; The

In [99]:
count = 0
for i, traj in mem1_objectives_dfs[16].groupby('traj_uid'):
    count += 2
    if count==3:
       break
print(traj.sort_values(by='step').full_trajectory_strings.iloc[1])

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
You will answer multiple complex questions using iterative reasoning, summarization, and web search.

At each step, you will see the questions, a cumulative summary of relevant information, the current search query, and search results (except in the first step, where only the questions are provided). Your task is to:

1. Perform reasoning and update a cumulative, concise summary within <think> ... </think>. This acts as persistent memory and must include all essential information from previous <think> and <information> tags.

2. Then choose one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The a

## belief compression over training

In [ ]:
num_objs = 16
temperature = '1.0'
length_penalty = '0.1'
dfraw = get_df_from_train_gen(f"/nas/ucb/jbjorner3/dev/optimal-explorer-dev/verl-agent/checkpoints/temp/nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_128sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_3/global_step_200_objectives_{str(num_objs)}_inference/train_gen.txt")
abbel_penalty_step200 = pd.DataFrame(dfraw.iloc[0].to_dict())

In [162]:
count = 0
for i, traj_df in abbel_penalty_step200.groupby('traj_uid'):
    if count == 0:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        print(traj_df['info'].iloc[0])
        break
    count += 1

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: What is the name of one the Italian Navy's new warship?; Carol Kane played the wife of which fictional character on Taxi?; Luis Gianneo was teacher of which chief exponent of Argentine folk music?; Who was one of the star in the South Korean horror romantic comedy film of Hwang In-ho?; What kind of group does 

In [ ]:
len("<answer> Fornara class; Oscar Madison; Atahualpa Yupanqui; Lee Byung-hun; punk rock band; grunge band; Jane Austen; Chicken Run; 2010; no; Somerset; Michael Bolton; Mr. Incredible; flagship; Erie; Alfred von Schlieffen </answer>")

In [166]:
count = 0
for i, traj_df in abbel_objectives_dfs_penalty[16].groupby('traj_uid'):
    if count == 3:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        break
    count += 1


system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: Where did John Athalarichos's father die?; Which film has the director born later, Passionate Summer or Little Presents?; What is the place of birth of the director of film The Strange Case Of Angelica?; Who was born later, Peter Samuel George Mackenzie or Josip Petrov Babich?; Where was the director of film B

In [ ]:
temperature = '1.0'
length_penalty = '0.05'
abbel_objectives_dfs_penalty05_16objs_ckpts ={}
for num_objs in [16]:
    for ckp in [60, 200]:
        dfraw = get_df_from_train_gen(eval_folder_path + f"nqhotpotqa_grpo_qwen2.5-7b-instruct_t_{temperature}_256sfr_seed1_sc_False_belief_prompting_True_ckpt_qwen2.5-7b-instruct_16sfr_seed1_sc_False_belief_prompting_True_is_mem1_False_belief_len_pen_{length_penalty}GRPO_INSTRUCT_4/global_step_{ckp}_objectives_{str(num_objs)}_inference3/train_gen.txt")
        abbel_objectives_dfs_penalty05_16objs_ckpts[ckp] = deduplicate_df(dfraw)

In [ ]:
count = 0
for i, traj_df in abbel_penalty_step200.groupby('traj_uid'):
    if count == 9:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        print(traj_df['info'].iloc[0])
        break
    count += 1

In [158]:
count = 0
for i, traj_df in abbel_objectives_dfs_penalty05_16objs_ckpts[60].groupby('traj_uid'):
    if count == 3:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        print(traj_df['info'].iloc[0])
        break
    count += 1

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: Where was the place of death of the director of film Earth (1957 Film)?; Where was the place of burial of the director of film The Tunnel (1933 German-Language Film)?; When did Thomson Mason (1759–1820)'s father die?; Which film whose director is younger, Welcome To Curiosity or A Night Of Adventure?; Are the 

In [159]:
count = 0
for i, traj_df in abbel_objectives_dfs_penalty05_16objs_ckpts[200].groupby('traj_uid'):
    if count == 3:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        print(traj_df['info'].iloc[0])
        break
    count += 1

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: who got the first nobel prize in physics?; when is the next deadpool movie being released?; which mode is used for short wave broadcast service?; the south west wind blows across nigeria between?; what does hp mean in war and order?; who wrote the first declaration of human rights?; who is the owner of reading

In [190]:
num_answers = []
for i, traj_df in abbel_objectives_dfs_penalty[16].groupby('traj_uid'):
    num_answers.append(traj_df.sort_values(by='step').response_ids_str.iloc[-1].split('<answer>')[-1].count(';'))
print(np.mean(num_answers))
num_answers.count(16)/len(num_answers)

9.50851656859709


0.11056054506039022

In [174]:
count = 0
for i, traj_df in abbel_objectives_dfs_penalty05[16].groupby('traj_uid'):
    if count == 7:
        print(traj_df.sort_values(by='step').full_trajectory_strings.iloc[0])
        print(traj_df['info'].iloc[0])
        break
    count += 1

system
You will answer multiple complex questions using iterative reasoning, and web search.
When taking an action, choose from one of the following actions:
   - If any question remains unanswered, issue a single query for one question inside <search> ... </search>. The query should consist of keywords or a short phrase. Only search one question at a time.
   - If all questions are answered, provide the final answers—separated by semicolons—within <answer> answer1; answer2; ... </answer>. The answers must be concise, contain only essential words, and avoid any explanations.

Important:
- Do not search multiple queries or questions simultaneously.

Answer the following questions: What is the religion of Averett University?; What is the religion of Kingdom of Sussex?; What is the religion of Cosmo Francesco Ruppi?; What is the religion of José Palmeira Lessa?; What is the religion of University of Evansville?; What is the religion of bishop?; What is the religion of St. Albert's Church,

# Getting Token Counts

In [3]:
# instantiate tokenizer
from verl.utils import hf_tokenizer
from verl.utils.fs import copy_to_local

MODEL_PATH='qwen/qwen2.5-7b-instruct'
local_path = copy_to_local(MODEL_PATH, use_shm=False)
tokenizer = hf_tokenizer(local_path, trust_remote_code=False, is_instruct_model=True)

In [15]:
import re

def extract_tag_text(text, tag):
    """Extract first instance of text between <tag></tag> tags"""
    if f"<{tag}>" in text and f"</{tag}>" in text:
        pattern = f'<{tag}>(.*?)</{tag}>'
        matches = re.findall(pattern, text, re.DOTALL)
        tagtext = matches[0].strip()
        subtracted_text = text.replace(f"<{tag}>{tagtext}</{tag}>", "")
    elif f"<{tag}>" in text and (tag in ["information", "environment"]) and (text[-len("\nassistant\n"):] == "\nassistant\n"): # search results got cut off and it goes straight to assistant
        print(f"missing </{tag}>")
        tagtext = text.split(f"<{tag}>",1)[1][:-len("\nassistant\n")] 
        subtracted_text = text.replace(f"<{tag}>{tagtext}","")
    elif (text.count(f"<{tag}>") != 1 or text.count(f"</{tag}>") != 1) and (tag == 'think') and '<search>' in text: # malformed internal state for MEM1
        print(f"0 or >1 opening and/or closing think tags, treating everything before <search> as the internal state")
        tagtext = text.split(f"<search>",1)[0] #MEM1 treats everything before <search> as the internal state
        subtracted_text = text.replace(tagtext, "")
    else:
        import pdb; pdb.set_trace()
        return "", text
    
    return tagtext, subtracted_text

def compute_peak_token_vanilla(traj_df, tokenizer) -> float:
    if traj_df.episode_rewards.max() == 0:
        return None
    full_traj_string = traj_df.full_trajectory_strings.iloc[0]
    output_str = full_traj_string.split('assistant\n', 1)[1] # remove initial instructions/system prompt as done in Zhou et al. MEM1 evals
    return len(tokenizer.encode(output_str))

def compute_peak_token_ABBEL(traj_df, tokenizer,memory_only=False) -> float:
    if traj_df.iloc[-1].rewards == 0:
        return None
    peak_count = 0
    for i,row in traj_df.iterrows():
        if row.step == 0:
            input_str, input_belief = "", ""
        else:
            input_str = row.input_ids_str.split('</instruction>\n', 1)[1] # remove initial instructions/system prompt as done in Zhou et al. MEM1 evals
            input_belief, input_str_extracted = extract_tag_text(input_str, 'belief')
            if row['info']['action_or_belief'] == 1: # belief update step
                input_action, input_str_extracted = extract_tag_text(input_str_extracted, 'search')
                input_observation, _ = extract_tag_text(input_str_extracted, 'environment')
                if '</hint>' in input_observation:
                    if len(input_observation.split('</hint>\n\n')) < 2:
                        import pdb; pdb.set_trace()
                    input_observation = input_observation.split('</hint>\n\n')[1]
                if len(input_observation) == 0:
                    import pdb; pdb.set_trace()
                input_str = input_belief + input_action + input_observation
            else:
                input_str = input_belief  
        output_str = row.response_ids_str
        if memory_only:
            peak_count = max(peak_count, len(tokenizer.encode(input_belief)))
        else:
            peak_count = max(peak_count, len(tokenizer.encode(input_str + output_str)))

    if peak_count == 0:
        return None

    return peak_count

def compute_peak_token_MEM1(traj_df, tokenizer,memory_only=False) -> float:
    if traj_df.iloc[-1].rewards == 0:
        return None
    peak_count = 0
    for i,row in traj_df.iterrows():
        if row.step == 0:
            input_str, input_IS = "", ""
        else:
            input_str = row.input_ids_str.split('\n\nassistant\n', 1)[1] # remove initial instructions/system prompt as done in Zhou et al. MEM1evals
            if '<search>' not in input_str:
                print('no search, skipping')
                return None
            input_IS, input_str_extracted = extract_tag_text(input_str, 'think') # previous internal state
            #input_IS = input_str.split('\n\n\nuser\n\n\n')[0] # previous internal state
            input_action, input_str_extracted = extract_tag_text(input_str_extracted, 'search')
            input_observation, _ = extract_tag_text(input_str_extracted, 'information')
            if len(input_observation) == 0:
                import pdb; pdb.set_trace()
            if '</hint>' in input_observation:
                input_observation = input_observation.split('</hint>\n\n')[1]
            input_str = input_IS + input_action + input_observation
        output_str = row.response_ids_str
        if memory_only:
            peak_count = max(peak_count, len(tokenizer.encode(input_IS)))
        else:
            peak_count = max(peak_count, len(tokenizer.encode(input_str + output_str)))

    if peak_count == 0:
        return None
    return peak_count


def compute_peak_token_all_trajectories(df, framework, tokenizer, memory_only=False) -> float:
# Create a dictionary of dataframes, one for each unique traj_uid
    all_peak_tokens = []
    for traj_uid, traj_df in df.groupby('traj_uid'):
        if framework == 'vanilla':
            all_peak_tokens.append(compute_peak_token_vanilla(traj_df, tokenizer))
        else:
            traj_df = traj_df.reset_index(drop=True).sort_values(by='step')
            if framework == 'ABBEL':
                all_peak_tokens.append(compute_peak_token_ABBEL(traj_df, tokenizer, memory_only))
            elif framework == 'MEM1':
                all_peak_tokens.append(compute_peak_token_MEM1(traj_df, tokenizer, memory_only))
        
      # filter out None
    all_peak_tokens = [x for x in all_peak_tokens if x is not None]
    return all_peak_tokens


In [5]:
def compute_scores(objectives_dfs):
    all_scores = []
    for num_objs in ALL_NUM_OBJS:
        traj_rewards = objectives_dfs[num_objs].groupby('traj_uid').episode_rewards.max()
        all_scores.append(traj_rewards)
    return all_scores

def compute_tokens(objectives_dfs, framework, tokenizer,memory_only=False,objs=ALL_NUM_OBJS):
    all_tokens = []
    for num_objs in objs:
        peak_tokens = compute_peak_token_all_trajectories(objectives_dfs[num_objs], framework, tokenizer, memory_only)
        all_tokens.append(peak_tokens)
        print("peak tokens for ", num_objs, "objs complete")
    return all_tokens

In [28]:
mtokens = compute_tokens(mem1_objectives_dfs, 'MEM1', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
missing </information>
missing opening and/or closing think tags
peak tokens for  4 objs complete
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
peak tokens for  8 objs complete
peak 

In [6]:
testd= {1:3,2:4,3:5}
for item in testd:
    print(item)

1
2
3


In [16]:
mtokens_seeds = []
for seed in mem1_seeds:
    mtokens = compute_tokens(mem1_seeds[seed], 'MEM1', tokenizer)
    mtokens_seeds.append(mtokens)

missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </i

In [ ]:
tokenspen01 = compute_tokens(abbel_objectives_dfs_penalty01, 'ABBEL', tokenizer)

In [104]:
tokenspen01s2 = compute_tokens(abbel_objectives_dfs_penalty01s2, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [110]:
tokenspen_seeds =[[np.mean(tokenspen01[i]), np.mean(tokenspen01s2[i])] for i in range(len(ALL_NUM_OBJS))]

In [30]:
tokens01 = compute_tokens(abbel_objectives_dfs01, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [ ]:
ztokens = compute_tokens(zabbel_objectives_dfs, 'ABBEL', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [8]:
vtokens = compute_tokens(vanilla_objectives_dfs, 'vanilla', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [160]:
v2tokens = compute_tokens(vanilla2_objectives_dfs, 'vanilla', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [ ]:
zvtokens = compute_tokens(zvanilla_objectives_dfs, 'vanilla', tokenizer)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [119]:
# Create separate legend figure
legend_fig = go.Figure()
# Add invisible traces to create legend

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='darkslategray'),
    name='ABBEL Zero'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='green'),
    name='ABBEL'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='yellowgreen'),
    name='ABBEL LP'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='purple'),
    name='VANILLA Zero'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='pink'),
    name='VANILLA'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='orange'),
    name='MEM1 Instruct'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='red'),
    name='MEM1 Base'
))

legend_fig.add_trace(go.Scatter(
    x=[None], y=[None],
    mode='markers',
    marker=dict(size=10, color='blue'),
    name='VANILLA 14B Zero'
))

legend_fig.update_layout(
    showlegend=True,
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=0.5,
        xanchor="center",
        x=0.5,
        font=dict(family='Times New Roman', size=16, color='black'),
        itemsizing='constant'
    ),
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    width=1000,
    height=100,
    margin=dict(l=0, r=0, t=0, b=0)
)

legend_fig.show()

legend_fig.write_image("notebooks/figures/QA_legend.pdf", width=1200, height=100)

In [62]:
import tempfile
import os

# Get the current temporary directory (optional)
print(f"Current temp dir: {tempfile.gettempdir()}")

# Set a new temporary directory
new_temp_dir = "notebooks/figures/tmp"
os.makedirs(new_temp_dir, exist_ok=True) # Create the directory if it doesn't exist
tempfile.tempdir = new_temp_dir

Current temp dir: /tmp


In [65]:
tempfile.tempdir = "/nas/ucb/dayan/optimal-explorer-dev/notebooks/figures/tmp"

In [64]:
print(f"Current temp dir: {tempfile.gettempdir()}")


Current temp dir: notebooks/figures/tmp


In [ ]:
# First plot - Bar chart for tokens
OBJS_TO_X = {1: 1, 2: 2.25, 4: 4, 8: 5.5, 16: 7}
fig = go.Figure()
bar_width = 0.13
bar_padding = 0.02
xoffsets = (np.arange(8) - 3) * (bar_width + bar_padding)

x_index = 0

# ABBEL Zero-Shot Tokens 
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] +xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in ztokens],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in ztokens]),
    name='ABBEL Zero-Shot',
    marker_color='darkslategray',
    width=bar_width
))

x_index += 1

# ABBEL Tokens 
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] +xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokens01],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokens01]),
    name='ABBEL',
    marker_color='green',
    width=bar_width
))

x_index += 1

# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in tokenspen01],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in tokenspen01]),
    name='ABBEL (Length Penalty)',
    marker_color='yellowgreen',
    width=bar_width
))

x_index += 1

# vanilla zero shot Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in zvtokens],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in zvtokens]),
    name='Vanilla Zero-Shot',
    marker_color='purple',
    width=bar_width
))

x_index += 1

# # Vanilla Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in v2tokens],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in v2tokens]),
    name='Vanilla',
    marker_color='pink',
    width=bar_width
))

x_index += 1

seeds_obj_result = []
for seed in mtokens_seeds:
    seeds_obj_result.append([np.mean(obj) for obj in seed])
seeds_obj_result = np.vstack(seeds_obj_result)
# MEM1 (Instruct) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=np.mean(seeds_obj_result,axis=0),
    error_y=dict(type='data', array=np.std(seeds_obj_result,axis=0)/np.sqrt(len(seeds_obj_result))),
    name='MEM1 (Instruct)',
    marker_color='orange',
    width=bar_width
))

x_index += 1

# MEM1 (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in REPORTED_OBJS],
    y=[640,801,1040],
    error_y=dict(type='data', array=[2,6,9]),
    name='MEM1',
    marker_color='red',
    width=bar_width
))

x_index += 1
# qwen 14b (Reported) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in REPORTED_OBJS],
    y=[1560,4470,3840],
    error_y=dict(type='data', array=[19,37,71]),
    name='Qwen2.5-14B-Instruct',
    marker_color='blue',
    width=bar_width
))

fig.update_layout(
    xaxis_title='# Objectives per Task',
    yaxis_title='Memory Usage (peak tokens)',
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=ALL_NUM_OBJS),
    yaxis=dict(type='log', tickvals=[500,600,700,800,1000,1200,1500,3000,4500,10000]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig.show()
#fig.write_image("notebooks/figures/QA_peak_tokens_fullevalv2.pdf", width=500, height=400)


In [48]:
internal_states = compute_tokens(mem1_objectives_dfs, 'MEM1', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
missing </information>
missing opening and/or closing think tags
peak tokens for  4 objs complete
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
missing opening and/or closing think tags
peak tokens for  8 objs complete
peak 

In [39]:
internal_states_seeds = []
for seed in mem1_seeds:
    internal_states_seeds.append(compute_tokens(mem1_seeds[seed], 'MEM1', tokenizer, memory_only=True))

missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </information>
missing </i

In [50]:
beliefs = compute_tokens(abbel_objectives_dfs01, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [51]:
beliefs_pen01 = compute_tokens(abbel_objectives_dfs_penalty01, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [117]:
beliefs_pen01s2 = compute_tokens(abbel_objectives_dfs_penalty01s2, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [120]:
beliefs_pen_seeds = [[np.mean(beliefs_pen01[i]), np.mean(beliefs_pen01s2[i])] for i in range(len(ALL_NUM_OBJS))]

In [42]:
zbeliefs = compute_tokens(zabbel_objectives_dfs, 'ABBEL', tokenizer, memory_only=True)

peak tokens for  1 objs complete
peak tokens for  2 objs complete
peak tokens for  4 objs complete
peak tokens for  8 objs complete
peak tokens for  16 objs complete


In [45]:
fig = go.Figure()
OBJS_TO_X = {1: 1, 2: 2, 4: 3, 8: 4, 16: 5}

bar_width = 0.14
bar_padding = 0.03
xoffsets = (np.arange(4) - 1.5) * (bar_width + bar_padding)

x_index = 0
# ABBEL Zero-Shot Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in zbeliefs],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in zbeliefs]),
    name='ABBEL Zero-Shot',
    marker_color='darkslategray',
    width=bar_width
))

x_index += 1
# ABBEL Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in beliefs],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in beliefs]),
    name='ABBEL',
    marker_color='green',
    width=bar_width
))

x_index += 1
# ABBEL (Length Penalty) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=[np.mean(t) for t in beliefs_pen01],
    error_y=dict(type='data', array=[np.std(t)/np.sqrt(len(t)) for t in beliefs_pen01]),
    name='ABBEL (Length Penalty)',
    marker_color='yellowgreen',
    width=bar_width
))

x_index += 1
seeds_obj_result = []
for seed in internal_states_seeds:
    seeds_obj_result.append([np.mean(obj) for obj in seed])
seeds_obj_result = np.vstack(seeds_obj_result)
# MEM1 (Instruct) Tokens
fig.add_trace(go.Bar(
    x=[OBJS_TO_X[o] + xoffsets[x_index] for o in ALL_NUM_OBJS],
    y=np.mean(seeds_obj_result,axis=0),
    error_y=dict(type='data', array=np.std(seeds_obj_result,axis=0)/np.sqrt(len(seeds_obj_result))),
    name='MEM1 (Instruct)',
    marker_color='orange',
    width=bar_width
))


fig.update_layout(
    xaxis_title='# Objectives per Task',
    yaxis_title='Belief Length (peak tokens)',
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    width=600,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=ALL_NUM_OBJS),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig.show()
#fig.write_image("figures/QA_peak_tokens_memory_only.pdf", width=500, height=400)
#fig.write_image("notebooks/figures/QA_peak_tokens_memory_only_fulleval_zeroshot.pdf", width=500, height=400)

In [ ]:
scores01 = compute_scores(abbel_objectives_dfs01)
scorespen01 = compute_scores(abbel_objectives_dfs_penalty01)
mscores = compute_scores(mem1_objectives_dfs)
vscores = compute_scores(vanilla_objectives_dfs)
zvscores = compute_scores(zvanilla_objectives_dfs)
zscores = compute_scores(zabbel_objectives_dfs)
scorespen01s2 = compute_scores(abbel_objectives_dfs_penalty01s2)
scorespen_seeds = {obj:[np.mean(scorespen01[obj]), np.mean(scorespen01s2[obj])] for obj in ALL_NUM_OBJS}

In [46]:
mscores_seeds = [compute_scores(mem1_seeds[seed]) for seed in mem1_seeds]

In [158]:
v2scores = compute_scores(vanilla2_objectives_dfs)

In [113]:
scorespen01s2 = compute_scores(abbel_objectives_dfs_penalty01s2)
scorespen_seeds = [[np.mean(scorespen01[obj]), np.mean(scorespen01s2[obj])] for obj in range(len(ALL_NUM_OBJS))]

In [49]:
# Second plot - Line chart for exact match scores
fig2 = go.Figure()
OBJS_TO_X = {1: 1, 2: 3, 4: 5, 8: 7, 16: 9}

# ABBEL Zero-Shot EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in zscores],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in zscores]),
    mode='markers+lines',
    name='ABBEL Zero-Shot',
    marker_color='darkslategray',
    marker_symbol='circle'
))

# ABBEL EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in scores01],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in scores01]),
    mode='markers+lines',
    name='ABBEL',
    marker_color='green',
    marker_symbol='circle'
))

# ABBEL penalty EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in scorespen01],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in scorespen01]),
    mode='markers+lines',
    name='ABBEL (Length Penalty 0.05)',
    marker_color='yellowgreen',
    marker_symbol='circle'
))

# Vanilla EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in v2scores],
    error_y=dict(type='data', array=[np.std(s)/np.sqrt(len(s)) for s in v2scores]),
    mode='markers+lines',
    name='Vanilla',
    marker_color='pink',
    marker_symbol='circle'
))

seeds_obj_result = []
for seed in mscores_seeds:
    seeds_obj_result.append([np.mean(obj) for obj in seed])
seeds_obj_result = np.vstack(seeds_obj_result)
# MEM1 (Instruct) EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=np.mean(seeds_obj_result,axis=0),
    error_y=dict(type='data', array=np.std(seeds_obj_result,axis=0)/np.sqrt(len(seeds_obj_result))),
    mode='markers+lines',
    name='MEM1 (Instruct)',
    marker_color='orange',
    marker_symbol='circle'
))


# Qwen 7B vanilla zero-shot
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in ALL_NUM_OBJS],
    y=[np.mean(s) for s in zvscores],
    mode='markers+lines',
    name='Qwen2.5-7B-Instruct',
    marker_color='purple',
    marker_symbol='circle',
    marker_size=10
))

# MEM1 (Reported) EM
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in REPORTED_OBJS],
    y=[0.709,1.87,1.97],
    mode='markers',
    name='MEM1',
    marker_color='red',
    marker_symbol='circle',
    marker_size=10
))

# Qwen 14B
fig2.add_trace(go.Scatter(
    x=[OBJS_TO_X[o] for o in REPORTED_OBJS],
    y=[0.732,1.55,0.567],
    mode='markers',
    name='Qwen2.5-14B-Instruct',
    marker_color='blue',
    marker_symbol='circle',
    marker_size=10
))


fig2.update_layout(
    template='plotly_white',
    font=dict(family='Times New Roman', size=16, color='black'),
    xaxis_title='# Objectives per Task',
    yaxis_title='Correct Answer Count (EM Score)',
    width=500,
    height=400,
    showlegend=False,
    xaxis=dict(tickvals=[0] + [OBJS_TO_X[o] for o in ALL_NUM_OBJS], ticktext=[0] + ALL_NUM_OBJS, range=[0, 9.5]),
    margin=dict(l=0, r=0, t=0, b=60)
)

fig2.show()
fig2.write_image("notebooks/figures/QA_scores_fulleval_pen01.pdf", width=500, height=400)


In [186]:
def print_stats(tokens, scores):
    stat_str = ""
    round_to = 2
    for num_objectives in [1,3,4]: # 2, 8 and 16 objectives
        stat_str += '& ' + str(round(np.mean(scores[num_objectives]), round_to))#+'$\pm $' + str(round(np.std(scores[num_objectives])/np.sqrt(len(scores[num_objectives])), round_to))
        stat_str += " "
        stat_str += '& ' + str(round(np.mean(np.array(tokens[num_objectives])*0.01), round_to))+'$\pm $' + str(round(np.std(np.array(tokens[num_objectives])*0.01)/np.sqrt(len(tokens[num_objectives])), round_to))
        stat_str += " "
    return stat_str

for tokens, scores in zip([mtokens, tokens01, tokenspen01, v2tokens, zvtokens, ztokens], [mscores, scores01, scorespen01, v2scores, zvscores, zscores]):
    print(print_stats(tokens, scores))

& 0.79 & 6.69$\pm $0.01 & 1.88 & 9.13$\pm $0.03 & 2.5 & 10.58$\pm $0.07 
& 0.73 & 6.78$\pm $0.01 & 2.4 & 8.95$\pm $0.03 & 3.57 & 10.12$\pm $0.06 
& 0.7 & 6.56$\pm $0.01 & 2.19 & 7.61$\pm $0.02 & 3.43 & 7.64$\pm $0.04 
& 0.79 & 17.87$\pm $0.15 & 2.54 & 64.07$\pm $0.44 & 3.06 & 96.08$\pm $0.37 
& 0.3 & 11.25$\pm $0.09 & 0.37 & 16.06$\pm $0.59 & 0.4 & 15.4$\pm $0.84 
& 0.53 & 6.85$\pm $0.01 & 1.28 & 8.67$\pm $0.04 & 1.62 & 9.46$\pm $0.08 


In [ ]:
with open("notebooks/saved_results/RL_evals_QA.pkl", "rb") as f:
    all_results_dict = pickle.load(f)
mtokens, tokenspen01, tokens01, v2tokens, zvtokens, ztokens, scores01, scorespen01, v2scores, zvscores, zscores, mscores, beliefs, beliefs_pen01, zbeliefs, internal_states = all_results_dict['mtokens'], all_results_dict['tokenspen01'], all_results_dict['tokens01'], all_results_dict['v2tokens'], all_results_dict['zvtokens'], all_results_dict['ztokens'], all_results_dict['scores01'], all_results_dict['scorespen01'], all_results_dict['v2scores'], all_results_dict['zvscores'], all_results_dict['zscores'], all_results_dict['mscores'], all_results_dict['beliefs'], all_results_dict['beliefs_pen01'], all_results_dict['zbeliefs'], all_results_dict['internal_states']

In [ ]:
all_results_dict = {}
all_results_dict['mscores'] = mscores
all_results_dict['scores01'] = scores01
all_results_dict['scorespen01'] = scorespen01
all_results_dict['tokens01'] = tokens01
all_results_dict['tokenspen01'] = tokenspen01
all_results_dict['mtokens'] = mtokens
all_results_dict['beliefs'] = beliefs
all_results_dict['beliefs_pen01'] = beliefs_pen01
all_results_dict['internal_states'] = internal_states
all_results_dict['vscores'] = vscores
all_results_dict['vtokens'] = vtokens
all_results_dict['zvscores'] = zvscores
all_results_dict['zvtokens'] = zvtokens
all_results_dict['zscores'] = zscores
all_results_dict['ztokens'] = ztokens

import pickle

with open("notebooks/saved_results/RL_evals_QA.pkl", "wb") as f:
    pickle.dump(all_results_dict, f)

In [50]:
all_results_dict['mscores_seeds'] = mscores_seeds
with open("notebooks/saved_results/RL_evals_QA.pkl", "wb") as f:
    pickle.dump(all_results_dict, f)

In [51]:
all_results_dict.keys()

dict_keys(['mscores', 'scores01', 'scorespen01', 'tokens01', 'tokenspen01', 'mtokens', 'beliefs', 'beliefs_pen01', 'internal_states', 'vscores', 'vtokens', 'zvscores', 'zvtokens', 'zscores', 'ztokens', 'zbeliefs', 'beliefs_pen01s2', 'scorespen01s2', 'tokenspen01s2', 'v2tokens', 'v2scores', 'mtokens_seeds', 'internal_states_seeds', 'mscores_seeds'])